# Pedal-aware eval — unattended runSynthesise with pedal → run the pipeline on all 50 → transcription baseline →pedal scoring. Roughly 105 minutes. Run All and leave it.### This notebook does not need the right kernelEvery heavy step shells out to `backend/venv/Scripts/python.exe`. The kernelitself only needs the standard library.That is the whole reason this was rewritten: the first version imported torch,librosa and the app services directly, so it only worked if Jupyter happened toselect the venv. With any default kernel it failed on the first real import —which is almost certainly the error seen when it was first tried.

In [ ]:
import os, sys, subprocess, pathlibROOT = pathlib.Path.cwd()while not (ROOT / 'backend').exists() and ROOT != ROOT.parent:    ROOT = ROOT.parentPY = ROOT / 'backend' / 'venv' / 'Scripts' / 'python.exe'print('repo  :', ROOT)print('python:', PY, '(exists)' if PY.exists() else '(MISSING — check the venv path)')def run(*args):    # -u so the child is not silent until it exits; an earlier run looked    # frozen for an hour purely because of stdout buffering.    p = subprocess.Popen([str(PY), '-u', *args], cwd=str(ROOT),                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,                         text=True, encoding='utf-8', errors='replace',                         env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})    for line in p.stdout:        print(line.rstrip(), flush=True)    p.wait()    print(f'[exit {p.returncode}]')    return p.returncode

## Everything, in one go  *(~105 min)*

In [ ]:
run('backend/eval/run_pedal_eval.py')

### Or step by step, if you want to stop between stages

In [ ]:
# run('backend/eval/synthesize_audio.py', '--force')            # ~50 min# run('backend/eval/run_pipeline.py', '--all', '--force')       # ~55 min# run('backend/eval/compare.py', '--all', '--save-baseline')    # fast

## Reading the results**`backend/eval/data/pedal_scores.json`** — the new measurement.The number that matters is `f1` against `f1_always_down`: truth is ~59% pedaldown, so a detector that simply held the pedal forever scores about **0.70**.Last run scored **0.7522**, a margin of only **+0.05** — which is whymusic-theory pedalling rules are on hold. Rules on top of a signal thatbarely beats a constant would mostly be rules on top of noise.**`backend/eval/baselines/baseline_*.json`** — transcription.Last run: COnP **0.9180**, COnPOff **0.1327**. Do not read COnP as animprovement over the old 0.7821 — the audio gained pedal *and* the pipelinegained the 25 ms floor fix at the same time, so the delta is unattributable.**COnPOff 0.1327 is the interesting one.** Offsets are now scored againstaudio that genuinely sustains, and it confirms the 2.10x offset inflationmeasured in `offset_inflation.py`. That is currently the largest measurabledefect in the pipeline.